# SentinelMail — Rule-Based DLP Benchmark

Evaluates `models/rule_based/RuleBasedDetector` (Microsoft Presidio) against the
ai4privacy validation split as a zero-training baseline for the ensemble benchmark.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from datasets import load_dataset
from tqdm.auto import tqdm

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

DATA_RAW = REPO_ROOT / "data" / "ai4privacy" / "raw"
RESULTS_DIR = REPO_ROOT / "evaluation" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

from data.preprocessing import build_labels, LABEL_COLS
from evaluation import (
    compute_all_metrics,
    classify_errors,
    error_summary,
    plot_confusion_matrices,
    plot_per_label_bars,
    save_results,
    from_dataframe,
    from_dicts,
)

## 1. Load Validation Split

In [ ]:
ds = load_dataset("ai4privacy/pii-masking-300k", cache_dir=str(DATA_RAW))
val_en = ds["validation"].filter(lambda x: x["language"] == "English")
val_df = val_en.to_pandas()
print(f"Validation rows (English): {len(val_df):,}")

## 2. Build Ground-Truth Labels

In [ ]:
label_df = val_df["privacy_mask"].apply(build_labels).apply(pd.Series)
val_df = pd.concat([val_df, label_df], axis=1)

print("Ground-truth label distribution (validation, EN):")
print(val_df[LABEL_COLS].sum().to_string())

## 3. Run Rule-Based Detector

In [ ]:
from models.rule_based import RuleBasedDetector

detector = RuleBasedDetector(score_threshold=0.4)
print("Detector loaded. Running on validation set...")

predictions = []
for text in tqdm(val_df["source_text"], desc="Detecting"):
    predictions.append(detector.predict(text))

pred_df = pd.DataFrame(predictions)
print("Predicted label distribution:")
print(pred_df[LABEL_COLS].sum().to_string())

## 4. Per-Label Metrics

In [ ]:
y_true = from_dataframe(val_df)
y_pred = from_dicts(predictions)

metrics = compute_all_metrics(y_true, y_pred)

print(f"Macro-F1: {metrics['macro_f1']:.4f}  "
      f"(95% CI: [{metrics['macro_f1_ci']['lower']:.4f}, {metrics['macro_f1_ci']['upper']:.4f}])")
print()

import pandas as pd
pd.DataFrame(metrics["per_label"]).T.round(4)

## 5. Confusion Matrices

In [ ]:
import matplotlib.pyplot as plt

cms, fig = plot_confusion_matrices(y_true, y_pred, return_fig=True)
fig.suptitle("Rule-Based Detector — Confusion Matrices (validation EN)", y=1.02)
plt.show()

_, bar_fig = plot_per_label_bars(metrics, return_fig=True)
plt.show()

## 6. Error Analysis

In [ ]:
texts = val_df["source_text"].tolist()

error_results = classify_errors(
    texts, y_true, y_pred,
    explain_fn=detector.explain,
    max_examples=20,
)

summary_df = error_summary(error_results)
print("Error counts by type and label:")
display(summary_df.pivot(index="error_type", columns="label", values="count").fillna(0).astype(int))

print("\n--- Sample FP Negations ---")
for r in error_results["FP_negation"][:5]:
    print(f"  [{r['label']}] {r['text_snippet'][:100]}")
    if r["explain"]:
        print(f"    detections: {[(h['entity_type'], h['text_snippet']) for h in r['explain']]}")

print("\n--- Sample FN Obfuscated ---")
for r in error_results["FN_obfuscated"][:5]:
    print(f"  [{r['label']}] {r['text_snippet'][:100]}")

## 7. Save Results

In [ ]:
out_path = RESULTS_DIR / "rule_based_metrics.json"

save_results(
    metrics,
    path=out_path,
    model_name="rule_based_presidio",
    n_samples=len(val_df),
    threshold=detector.score_threshold,
    extra_metadata={"score_threshold": detector.score_threshold},
)

print(f"Saved to {out_path}")